In [2]:
import requests
import pandas as pd
import os
import time
from dotenv import load_dotenv

In [3]:
load_dotenv()
github_key = os.getenv("GITHUB_TOKEN")

headers = {
        "Accept": "application/vnd.github+json",
        "Authorization": f"Bearer {github_key}"
}

url = "https://api.github.com/repos/nodejs/node/pulls?state=all"
# response = requests.get(url, headers=headers)
# data = response.json()

In [4]:
all_results = []
cont = 0
while url:
    cont += 1
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        print(response.headers.get("Link", ""))
        data = response.json()
        
        all_results.extend(data)

        link_header = response.headers.get("Link", "")
        next_url = None
        for link in link_header.split(","):
            if 'rel="next"' in link:
                next_url = link[link.find("<")+1:link.find(">")]
                break

        url = next_url

        if cont == 30:
            url = None

        # 'don't get banned' check
        time.sleep(1)
        
    elif response.status_code == 202:
        print("Compiling data, try again shortly")
        break
    else:
        print(f"Error: {response.status_code}")
        break

<https://api.github.com/repositories/27193779/pulls?state=all&page=2>; rel="next", <https://api.github.com/repositories/27193779/pulls?state=all&page=1229>; rel="last"
<https://api.github.com/repositories/27193779/pulls?state=all&page=1>; rel="prev", <https://api.github.com/repositories/27193779/pulls?state=all&page=3>; rel="next", <https://api.github.com/repositories/27193779/pulls?state=all&page=1229>; rel="last", <https://api.github.com/repositories/27193779/pulls?state=all&page=1>; rel="first"
<https://api.github.com/repositories/27193779/pulls?state=all&page=2>; rel="prev", <https://api.github.com/repositories/27193779/pulls?state=all&page=4>; rel="next", <https://api.github.com/repositories/27193779/pulls?state=all&page=1229>; rel="last", <https://api.github.com/repositories/27193779/pulls?state=all&page=1>; rel="first"
<https://api.github.com/repositories/27193779/pulls?state=all&page=3>; rel="prev", <https://api.github.com/repositories/27193779/pulls?state=all&page=5>; rel="nex

In [5]:
df = pd.json_normalize(
            all_results, 
            record_path=None, 
            meta=None, 
            errors='ignore'
        )
df.tail(100).to_json("../data/dataset/test/testo.json", orient="records")

In [6]:
cleaned_df = df.dropna(axis=1, how='all')
filtered_df = cleaned_df[["number", "state", "title", "body", "locked", "created_at", "updated_at", "closed_at", "merged_at", "assignees", "user.login", "labels", "author_association", "user.repos_url", "user.followers_url", "user.organizations_url", "user.starred_url", "user.type", "base.user.login", 'base.repo.name', "base.user.followers_url", "base.user.starred_url", "base.repo.created_at", "base.repo.updated_at", "base.repo.pushed_at", "base.repo.size", "base.repo.releases_url", "base.repo.stargazers_count", "base.repo.watchers_count", "base.repo.language", "base.repo.has_issues", "base.repo.has_projects", "base.repo.has_downloads", "base.repo.has_wiki", "base.repo.has_pages", "base.repo.has_discussions", "base.repo.forks_count"]]

In [7]:
non_url_columns_df = filtered_df.drop(columns=[col for col in filtered_df.columns if col.endswith('_url')])

In [8]:
non_url_columns_df.columns = ['number', 'state', 'title', 'body', 'locked', 'created_at', 'updated_at', 'closed_at', 'merged_at', 'assignees', 'user_name','labels','author_association','user_type','repo_owner_name', 'repo_name','repo_created_at','repo_updated_at','repo_pushed_at','repo_size','repo_stargazer_count','repo_watcher_count','repo_language', 'repo_has_issue', 'repo_has_projects', 'repo_has_downloads', 'repo_has_wiki', 'repo_has_pages', 'repo_has_discussions', 'repo_fork_count']

In [10]:
non_url_columns_df.to_csv("../data/dataset/pull_request_data.csv", index=False)
non_url_columns_df

,number,state,title,body,locked,created_at,updated_at,closed_at,merged_at,assignees,...,repo_stargazer_count,repo_watcher_count,repo_language,repo_has_issue,repo_has_projects,repo_has_downloads,repo_has_wiki,repo_has_pages,repo_has_discussions,repo_fork_count
0,58339,open,doc: add latest security release steward,As titled,False,2025-05-14T21:51:54Z,2025-05-14T21:52:13Z,None,None,[],...,111253,111253,JavaScript,True,True,True,False,False,False,31553
1,58337,open,lib: deprecate `_stream_*` modules,Runtime deprecation of all the `_stream_*` mod...,False,2025-05-14T18:38:39Z,2025-05-14T21:23:54Z,None,None,[],...,111253,111253,JavaScript,True,True,True,False,False,False,31553
2,58336,closed,Mock usdt,Mock usdt\r\n\r\n<!--\r\nBefore submitting a p...,False,2025-05-14T17:30:50Z,2025-05-14T17:59:06Z,2025-05-14T17:59:06Z,None,[],...,111253,111253,JavaScript,True,True,True,False,False,False,31553
3,58335,open,tools: add missing highway defines for IBM i,This was added for AIX but should have include...,False,2025-05-14T16:50:21Z,2025-05-14T20:34:08Z,None,None,[],...,111253,111253,JavaScript,True,True,True,False,False,False,31553
4,58334,open,doc: clarify --watch and --watch-path usage wi...,This PR updates the --run CLI documentation to...,False,2025-05-14T16:02:41Z,2025-05-14T16:08:33Z,None,None,[],...,111253,111253,JavaScript,True,True,True,False,False,False,31553
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
895,57034,closed,test: fix test-without-async-context-frame.mjs...,The test is spawning the python test runner di...,False,2025-02-13T15:31:04Z,2025-02-15T19:37:41Z,2025-02-15T19:37:31Z,2025-02-15T19:37:31Z,[],...,111253,111253,JavaScript,True,True,True,False,False,False,31553
896,57031,closed,src: lock the isolate properly in IsolateData ...,Otherwise it may fail the DCHECK that uses the...,False,2025-02-13T14:28:48Z,2025-02-15T23:49:49Z,2025-02-15T23:49:39Z,2025-02-15T23:49:39Z,[],...,111253,111253,JavaScript,True,True,True,False,False,False,31553
897,57029,open,doc: correct definition of v8.getHeapStatistic...,"<!--\r\nBefore submitting a pull request, plea...",False,2025-02-13T13:21:59Z,2025-04-02T09:54:12Z,None,None,[],...,111253,111253,JavaScript,True,True,True,False,False,False,31553
898,57028,closed,doc: recommend writing tests in new files and ...,The previous phrasing encouraged or did not di...,False,2025-02-13T12:49:20Z,2025-02-15T12:58:12Z,2025-02-15T12:58:09Z,2025-02-15T12:58:09Z,[],...,111253,111253,JavaScript,True,True,True,False,False,False,31553


In [27]:
url_columns = filtered_df.filter(regex='_url$')
url_columns

,user.repos_url,user.followers_url,user.organizations_url,user.starred_url,base.user.followers_url,base.user.starred_url,base.repo.releases_url
0,https://api.github.com/users/vagostep/repos,https://api.github.com/users/vagostep/followers,https://api.github.com/users/vagostep/orgs,https://api.github.com/users/vagostep/starred{...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
1,https://api.github.com/users/danmcd/repos,https://api.github.com/users/danmcd/followers,https://api.github.com/users/danmcd/orgs,https://api.github.com/users/danmcd/starred{/o...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
2,https://api.github.com/users/theoludwig/repos,https://api.github.com/users/theoludwig/followers,https://api.github.com/users/theoludwig/orgs,https://api.github.com/users/theoludwig/starre...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
3,https://api.github.com/users/StefanStojanovic/...,https://api.github.com/users/StefanStojanovic/...,https://api.github.com/users/StefanStojanovic/...,https://api.github.com/users/StefanStojanovic/...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
4,https://api.github.com/users/LiviaMedeiros/repos,https://api.github.com/users/LiviaMedeiros/fol...,https://api.github.com/users/LiviaMedeiros/orgs,https://api.github.com/users/LiviaMedeiros/sta...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
...,...,...,...,...,...,...,...
895,https://api.github.com/users/joyeecheung/repos,https://api.github.com/users/joyeecheung/follo...,https://api.github.com/users/joyeecheung/orgs,https://api.github.com/users/joyeecheung/starr...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
896,https://api.github.com/users/aduh95/repos,https://api.github.com/users/aduh95/followers,https://api.github.com/users/aduh95/orgs,https://api.github.com/users/aduh95/starred{/o...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
897,https://api.github.com/users/aduh95/repos,https://api.github.com/users/aduh95/followers,https://api.github.com/users/aduh95/orgs,https://api.github.com/users/aduh95/starred{/o...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...
898,https://api.github.com/users/aduh95/repos,https://api.github.com/users/aduh95/followers,https://api.github.com/users/aduh95/orgs,https://api.github.com/users/aduh95/starred{/o...,https://api.github.com/users/nodejs/followers,https://api.github.com/users/nodejs/starred{/o...,https://api.github.com/repos/nodejs/node/relea...


In [30]:
def fetch_url(url):
    try:
        print(url)
        response = requests.get(url)
        response.raise_for_status()
        return response.json()  # or response.text if it's not JSON
    except requests.RequestException as e:
        return f'Error: {e}'

# For each column, fetch and replace URLs with the data
count = 0
for col in url_columns.columns:
    print(f'col: {col}')
    count += 1
    url_columns[col] = url_columns[col].apply(fetch_url)
    time.sleep(1)
    if count == 30:
        break

col: user.repos_url
https://api.github.com/users/vagostep/repos
https://api.github.com/users/danmcd/repos
https://api.github.com/users/theoludwig/repos
https://api.github.com/users/StefanStojanovic/repos
https://api.github.com/users/LiviaMedeiros/repos
https://api.github.com/users/targos/repos
https://api.github.com/users/targos/repos
https://api.github.com/users/szegedi/repos
https://api.github.com/users/aduh95/repos
https://api.github.com/users/github-actions%5Bbot%5D/repos
https://api.github.com/users/puskin/repos
https://api.github.com/users/avivkeller/repos
https://api.github.com/users/mertcanaltin/repos
https://api.github.com/users/panva/repos
https://api.github.com/users/jazelly/repos
https://api.github.com/users/NishaGadave/repos
https://api.github.com/users/Skc-VitInProjects/repos
https://api.github.com/users/khardix/repos
https://api.github.com/users/lpinca/repos
https://api.github.com/users/panva/repos
https://api.github.com/users/jasnell/repos
https://api.github.com/users/j

KeyboardInterrupt: 